In [ ]:
#@title 🧬 rubiksat — SAT Solver via Generalized Rubik's Cube Reduction (Sparse Permutation, C++ Accelerated)
#@markdown Sube un .cnf / .cnf.xz / .cnf.gz / .cnf.bz2 o usa el ejemplo integrado.
#@markdown Output: <nombre>_dynamics.txt + resultado en pantalla.
#@markdown ---
#@markdown Optimización clave: El cubo N×N×N tiene 6N² stickers pero
#@markdown la codificación SAT solo desplaza O(n·m·N) de ellos. Usamos
#@markdown representación de permutación esparsa — solo rastreamos
#@markdown stickers no-identidad. Cada operación es O(N) en vez de O(N²).
#@markdown
#@markdown **v2 Optimizations**: Pass-by-reference everywhere, flat arrays
#@markdown with robin-hood hashing, pre-allocated cycle buffers, zero-copy
#@markdown delta computation, LTO + PGO-ready release build flags.

import os, sys, time, ctypes, tempfile, subprocess
from datetime import datetime
from typing import List, Dict
from google.colab import files as colab_files

# ══════════════════════════════════════════════════════════════
# C++ CORE — Optimized Sparse Permutation Representation
# ══════════════════════════════════════════════════════════════
# Key optimizations vs v1:
# 1. ALL large structures passed by const reference, never by value
# 2. Replaced std::unordered_map with flat open-addressing hash map
#    (cache-friendly, no heap allocation per entry)
# 3. Pre-allocated cycle buffers — no per-call vector allocations
# 4. Delta computation is fully in-place, no SparseCube copy needed
#    for single-move trials
# 5. Face rotation uses direct cycle chasing, no intermediate sets
# 6. Compile with -O3 -march=native -flto -DNDEBUG

CPP_SOURCE = r"""
#include <cstdint>
#include <cstring>
#include <cstdlib>
#include <cstdio>
#include <vector>
#include <algorithm>
#include <utility>
#include <cassert>

extern "C" {

// ══════════════════════════════════════════════════════════════
// FLAT OPEN-ADDRESSING HASH MAP (cache-friendly, no heap alloc per entry)
// ══════════════════════════════════════════════════════════════
// Replaces std::unordered_map<int,int> which:
//   - allocates a node per insertion (heap pressure)
//   - has pointer-chasing on lookup (cache misses)
//   - copies are O(n) with O(n) allocations
// This flat map uses open addressing with linear probing:
//   - all data in one contiguous array (cache-friendly)
//   - copy is a single memcpy
//   - no per-entry heap allocation

struct FlatMap {
    struct Entry {
        int key;
        int val;
        uint8_t occupied; // 0=empty, 1=occupied, 2=tombstone
    };

    Entry* data;
    int capacity;
    int size_;
    int mask;

    FlatMap() : data(nullptr), capacity(0), size_(0), mask(0) {}

    void init(int cap) {
        // Round up to power of 2
        int c = 16;
        while (c < cap * 2) c <<= 1; // load factor < 0.5
        capacity = c;
        mask = c - 1;
        size_ = 0;
        data = (Entry*)calloc(c, sizeof(Entry));
    }

    void destroy() {
        if (data) { free(data); data = nullptr; }
        size_ = 0;
    }

    // Copy from another FlatMap — single memcpy, no per-entry alloc
    void copy_from(const FlatMap& other) {
        if (data) free(data);
        capacity = other.capacity;
        mask = other.mask;
        size_ = other.size_;
        data = (Entry*)malloc(capacity * sizeof(Entry));
        memcpy(data, other.data, capacity * sizeof(Entry));
    }

    static inline int hash_(int key) {
        // Fast integer hash (murmurhash3 finalizer)
        unsigned int h = (unsigned int)key;
        h ^= h >> 16;
        h *= 0x45d9f3b;
        h ^= h >> 16;
        return (int)h;
    }

    // Returns pointer to value if found, nullptr otherwise
    int* find(int key) const {
        if (!data) return nullptr;
        int idx = hash_(key) & mask;
        for (int i = 0; i < capacity; i++) {
            int pos = (idx + i) & mask;
            if (data[pos].occupied == 0) return nullptr; // empty slot
            if (data[pos].occupied == 1 && data[pos].key == key)
                return &data[pos].val;
        }
        return nullptr;
    }

    void set(int key, int val) {
        if (!data) init(64);
        // Check if need rehash (load > 0.6)
        if (size_ * 5 > capacity * 3) rehash();

        int idx = hash_(key) & mask;
        int first_tomb = -1;
        for (int i = 0; i < capacity; i++) {
            int pos = (idx + i) & mask;
            if (data[pos].occupied == 0) {
                // Empty slot — use tombstone if we found one, else this slot
                int ins = (first_tomb >= 0) ? first_tomb : pos;
                data[ins].key = key;
                data[ins].val = val;
                data[ins].occupied = 1;
                size_++;
                return;
            }
            if (data[pos].occupied == 2 && first_tomb < 0) {
                first_tomb = pos;
            }
            if (data[pos].occupied == 1 && data[pos].key == key) {
                data[pos].val = val;
                return; // update existing
            }
        }
        // Should not reach here if load factor is maintained
        rehash();
        set(key, val);
    }

    bool erase(int key) {
        if (!data) return false;
        int idx = hash_(key) & mask;
        for (int i = 0; i < capacity; i++) {
            int pos = (idx + i) & mask;
            if (data[pos].occupied == 0) return false;
            if (data[pos].occupied == 1 && data[pos].key == key) {
                data[pos].occupied = 2; // tombstone
                size_--;
                return true;
            }
        }
        return false;
    }

    bool empty() const { return size_ == 0; }
    int size() const { return size_; }

    void rehash() {
        int old_cap = capacity;
        Entry* old_data = data;
        int new_cap = (capacity == 0) ? 64 : capacity * 2;
        capacity = new_cap;
        mask = new_cap - 1;
        data = (Entry*)calloc(new_cap, sizeof(Entry));
        size_ = 0;
        if (old_data) {
            for (int i = 0; i < old_cap; i++) {
                if (old_data[i].occupied == 1)
                    set(old_data[i].key, old_data[i].val);
            }
            free(old_data);
        }
    }

    // Iterator-like: call func for each (key, val) pair
    template<typename F>
    void for_each(F&& func) const {
        if (!data) return;
        for (int i = 0; i < capacity; i++) {
            if (data[i].occupied == 1)
                func(data[i].key, data[i].val);
        }
    }

    // Collect keys matching a predicate into a pre-allocated buffer
    // Returns count written
    int collect_keys_in_range(int lo, int hi, int* buf, int buf_cap) const {
        if (!data) return 0;
        int cnt = 0;
        for (int i = 0; i < capacity && cnt < buf_cap; i++) {
            if (data[i].occupied == 1 && data[i].key >= lo && data[i].key < hi)
                buf[cnt++] = data[i].key;
        }
        return cnt;
    }
};


// ══════════════════════════════════════════════════════════════
// SPARSE CUBE — using FlatMap instead of std::unordered_map
// ══════════════════════════════════════════════════════════════

struct SparseCube {
    int n;
    FlatMap perm;
};

static inline int pos(int n, int face, int r, int c) {
    return face * n * n + r * n + c;
}

// Get sticker value — NEVER copies the cube
static inline int sget(const SparseCube* sc, int p) {
    int* v = sc->perm.find(p);
    return v ? *v : p;
}

// Set sticker value — in-place mutation
static inline void sset(SparseCube* sc, int p, int val) {
    if (val == p) {
        sc->perm.erase(p);
    } else {
        sc->perm.set(p, val);
    }
}

SparseCube* scube_create(int n) {
    SparseCube* sc = new SparseCube;
    sc->n = n;
    // Pre-allocate for expected sparsity
    sc->perm.init(std::max(256, n * 16));
    return sc;
}

SparseCube* scube_copy(const SparseCube* src) {
    SparseCube* sc = new SparseCube;
    sc->n = src->n;
    sc->perm.copy_from(src->perm); // single memcpy, no per-entry alloc
    return sc;
}

void scube_destroy(SparseCube* sc) {
    if (sc) {
        sc->perm.destroy();
        delete sc;
    }
}

int scube_solved(const SparseCube* sc) {
    return sc->perm.empty() ? 1 : 0;
}

int scube_misplaced(const SparseCube* sc) {
    return sc->perm.size();
}


// ══════════════════════════════════════════════════════════════
// FACE ROTATION — optimized: direct cycle chasing, no sets/vectors
// ══════════════════════════════════════════════════════════════

static void srot_face(SparseCube* sc, int fi, int k) {
    k = ((k % 4) + 4) % 4;
    if (k == 0) return;
    int n = sc->n;
    int base = fi * n * n;

    // Collect non-identity positions on this face into a stack buffer
    // Use a fixed-size buffer on the stack for small faces,
    // heap-allocate only if needed
    int stack_buf[512];
    int* face_pos = stack_buf;
    int face_cnt = sc->perm.collect_keys_in_range(base, base + n * n, stack_buf, 512);

    // If more than 512, heap allocate (very rare)
    int* heap_buf = nullptr;
    if (face_cnt >= 512) {
        heap_buf = new int[n * n];
        face_cnt = sc->perm.collect_keys_in_range(base, base + n * n, heap_buf, n * n);
        face_pos = heap_buf;
    }

    if (face_cnt == 0) { if (heap_buf) delete[] heap_buf; return; }

    // Track which positions we've visited in cycles
    // Use a bitset on the stack for moderate n, or a flat array
    // For n up to ~256 (65536 positions per face), a flat bool array is fine
    bool stack_visited[4096];
    bool* visited = stack_visited;
    bool* heap_visited = nullptr;
    if (n * n > 4096) {
        heap_visited = new bool[n * n]();
        visited = heap_visited;
    } else {
        memset(visited, 0, n * n * sizeof(bool));
    }

    // For each non-identity face position, chase its 4-cycle
    // Also need to include all positions IN the cycle even if they're identity
    for (int fi_idx = 0; fi_idx < face_cnt; fi_idx++) {
        int p = face_pos[fi_idx];
        int local = p - base;
        if (visited[local]) continue;

        // Chase the 4-cycle: (r,c) -> (c, n-1-r) -> ...
        int cyc[4];
        cyc[0] = local;
        for (int s = 1; s < 4; s++) {
            int rr = cyc[s-1] / n, cc = cyc[s-1] % n;
            cyc[s] = cc * n + (n - 1 - rr);
        }

        // Mark all visited
        for (int s = 0; s < 4; s++) visited[cyc[s]] = true;

        // Check for degenerate cycle (center of odd face)
        if (cyc[0] == cyc[1]) continue;

        // Read current values
        int vals[4];
        for (int s = 0; s < 4; s++) vals[s] = sget(sc, base + cyc[s]);

        // Apply k CW rotations: position cyc[s] gets value from cyc[(s - k + 4) % 4]
        for (int s = 0; s < 4; s++) {
            int src = (s - k + 4) % 4;
            sset(sc, base + cyc[s], vals[src]);
        }
    }

    if (heap_buf) delete[] heap_buf;
    if (heap_visited) delete[] heap_visited;
}


// ══════════════════════════════════════════════════════════════
// SLICE ROTATION — no vectors allocated per call
// ══════════════════════════════════════════════════════════════

void scube_rot(SparseCube* sc, int ax, int idx, int d) {
    int n = sc->n;

    // Normalize direction: CCW = 3x CW
    int ncycles = (d == 1) ? 1 : 3;

    if (ax == 0) { // x-slice, column col=idx
        int col = idx;
        for (int t = 0; t < ncycles; t++) {
            for (int r = 0; r < n; r++) {
                int p0 = pos(n, 2, r, col);
                int p1 = pos(n, 4, r, col);
                int p2 = pos(n, 3, n-1-r, n-1-col);
                int p3 = pos(n, 5, r, col);

                int v0 = sget(sc, p0);
                int v1 = sget(sc, p1);
                int v2 = sget(sc, p2);
                int v3 = sget(sc, p3);

                sset(sc, p0, v3);
                sset(sc, p1, v0);
                sset(sc, p2, v1);
                sset(sc, p3, v2);
            }
        }
        if (col == 0) srot_face(sc, 1, -d);
        else if (col == n-1) srot_face(sc, 0, d);
    }
    else if (ax == 1) { // y-slice, row r=idx
        int r = idx;
        for (int t = 0; t < ncycles; t++) {
            for (int c = 0; c < n; c++) {
                int p0 = pos(n, 0, r, c);
                int p1 = pos(n, 5, r, c);
                int p2 = pos(n, 1, r, c);
                int p3 = pos(n, 4, r, c);

                int v0 = sget(sc, p0);
                int v1 = sget(sc, p1);
                int v2 = sget(sc, p2);
                int v3 = sget(sc, p3);

                sset(sc, p0, v1);
                sset(sc, p1, v2);
                sset(sc, p2, v3);
                sset(sc, p3, v0);
            }
        }
        if (r == 0) srot_face(sc, 3, -d);
        else if (r == n-1) srot_face(sc, 2, d);
    }
    else { // z-slice, layer=idx
        int layer = idx;
        int r = n - 1 - layer;
        for (int t = 0; t < ncycles; t++) {
            for (int c = 0; c < n; c++) {
                int pA = pos(n, 0, r, c);
                int pB = pos(n, 2, r, c);
                int pC = pos(n, 1, n-1-r, n-1-c);
                int pD = pos(n, 3, r, c);

                int vA = sget(sc, pA);
                int vB = sget(sc, pB);
                int vC = sget(sc, pC);
                int vD = sget(sc, pD);

                sset(sc, pA, vD);
                sset(sc, pB, vA);
                sset(sc, pC, vB);
                sset(sc, pD, vC);
            }
        }
        if (layer == 0) srot_face(sc, 5, d);
        else if (layer == n-1) srot_face(sc, 4, d);
    }
}


// ══════════════════════════════════════════════════════════════
// ENCODER — SAT → Cube Configuration C_φ
// ══════════════════════════════════════════════════════════════

SparseCube* encode_sat(int nv, int nc, const int* clause_offsets,
                       const int* clause_lits, int total_lits, int dim) {
    SparseCube* c = scube_create(dim);

    for (int i = 1; i <= nv; i++) {
        int sp = 2 * i;
        int sn = 2 * i - 1;
        if (sp < dim) scube_rot(c, 0, sp, 1);
        if (sn < dim) scube_rot(c, 0, sn, -1);
    }

    for (int j = 0; j < nc; j++) {
        int qj = 2 * nv + 2 * (j + 1);
        if (qj >= dim) continue;

        int start = clause_offsets[j];
        int end = (j + 1 < nc) ? clause_offsets[j + 1] : total_lits;

        for (int k = start; k < end; k++) {
            int lit = clause_lits[k];
            int var = lit > 0 ? lit : -lit;
            if (var < 1 || var > nv) continue;

            int sp = 2 * var;
            int sn = 2 * var - 1;

            if (lit > 0) {
                if (sp < dim) scube_rot(c, 0, sp, 1);
                if (sn < dim) scube_rot(c, 0, sn, -1);
                scube_rot(c, 1, qj, 1);
                if (sn < dim) scube_rot(c, 0, sn, 1);
                if (sp < dim) scube_rot(c, 0, sp, -1);
            } else {
                if (sn < dim) scube_rot(c, 0, sn, 1);
                if (sp < dim) scube_rot(c, 0, sp, -1);
                scube_rot(c, 1, qj, -1);
                if (sp < dim) scube_rot(c, 0, sp, 1);
                if (sn < dim) scube_rot(c, 0, sn, -1);
            }
        }
    }

    return c;
}


// ══════════════════════════════════════════════════════════════
// DELTA COMPUTATION — zero-copy, O(N) per trial move
// ══════════════════════════════════════════════════════════════
// KEY OPTIMIZATION: We never copy the SparseCube for single-move
// delta evaluation. We compute the change in misplaced count
// purely from reading the current state.

// 4-cycle structure — stack-allocated, no heap
struct Cycle4 { int p[4]; };

// Build slice cycles into a pre-allocated buffer. Returns count.
// Buffer must have space for at least N cycles.
static int build_slice_cycles(int n, int ax, int idx, Cycle4* buf) {
    int cnt = 0;
    if (ax == 0) {
        int col = idx;
        for (int r = 0; r < n; r++) {
            buf[cnt].p[0] = pos(n, 2, r, col);
            buf[cnt].p[1] = pos(n, 4, r, col);
            buf[cnt].p[2] = pos(n, 3, n-1-r, n-1-col);
            buf[cnt].p[3] = pos(n, 5, r, col);
            cnt++;
        }
    } else if (ax == 1) {
        int r = idx;
        for (int c = 0; c < n; c++) {
            buf[cnt].p[0] = pos(n, 0, r, c);
            buf[cnt].p[1] = pos(n, 5, r, c);
            buf[cnt].p[2] = pos(n, 1, r, c);
            buf[cnt].p[3] = pos(n, 4, r, c);
            cnt++;
        }
    } else {
        int layer = idx;
        int r = n - 1 - layer;
        for (int c = 0; c < n; c++) {
            buf[cnt].p[0] = pos(n, 0, r, c);
            buf[cnt].p[1] = pos(n, 2, r, c);
            buf[cnt].p[2] = pos(n, 1, n-1-r, n-1-c);
            buf[cnt].p[3] = pos(n, 3, r, c);
            cnt++;
        }
    }
    return cnt;
}

// Compute delta misplaced for a single CW or CCW slice rotation
// WITHOUT copying the cube — pure read-only computation
static int delta_misplaced_slice_only(const SparseCube* sc, int n,
                                       const Cycle4* cycles, int ncyc, int d) {
    int delta = 0;
    for (int ci = 0; ci < ncyc; ci++) {
        const int* p = cycles[ci].p;
        int v0 = sget(sc, p[0]);
        int v1 = sget(sc, p[1]);
        int v2 = sget(sc, p[2]);
        int v3 = sget(sc, p[3]);

        int old_mis = (v0 != p[0]) + (v1 != p[1]) + (v2 != p[2]) + (v3 != p[3]);

        int nv0, nv1, nv2, nv3;
        if (d == 1) {
            nv0 = v3; nv1 = v0; nv2 = v1; nv3 = v2;
        } else {
            nv0 = v1; nv1 = v2; nv2 = v3; nv3 = v0;
        }

        int new_mis = (nv0 != p[0]) + (nv1 != p[1]) + (nv2 != p[2]) + (nv3 != p[3]);
        delta += (new_mis - old_mis);
    }
    return delta;
}

// Compute delta for face rotation (boundary slices only)
static int delta_face_rotation(const SparseCube* sc, int n, int face_fi, int face_k) {
    if (face_fi < 0 || face_k == 0) return 0;

    int base = face_fi * n * n;
    int delta = 0;

    // Collect face positions from perm
    int stack_buf[512];
    int* face_pos = stack_buf;
    int face_cnt = sc->perm.collect_keys_in_range(base, base + n * n, stack_buf, 512);
    int* heap_buf = nullptr;
    if (face_cnt >= 512) {
        heap_buf = new int[n * n];
        face_cnt = sc->perm.collect_keys_in_range(base, base + n * n, heap_buf, n * n);
        face_pos = heap_buf;
    }

    if (face_cnt == 0) { if (heap_buf) delete[] heap_buf; return 0; }

    // Visited array on stack
    bool stack_vis[4096];
    bool* visited = stack_vis;
    bool* heap_vis = nullptr;
    if (n * n > 4096) {
        heap_vis = new bool[n * n]();
        visited = heap_vis;
    } else {
        memset(visited, 0, n * n * sizeof(bool));
    }

    for (int fi_idx = 0; fi_idx < face_cnt; fi_idx++) {
        int local = face_pos[fi_idx] - base;
        if (visited[local]) continue;

        int cyc[4];
        cyc[0] = local;
        for (int s = 1; s < 4; s++) {
            int rr = cyc[s-1] / n, cc = cyc[s-1] % n;
            cyc[s] = cc * n + (n - 1 - rr);
        }
        for (int s = 0; s < 4; s++) visited[cyc[s]] = true;
        if (cyc[0] == cyc[1]) continue;

        int vals[4];
        for (int s = 0; s < 4; s++) vals[s] = sget(sc, base + cyc[s]);

        int old_m = 0;
        for (int s = 0; s < 4; s++)
            if (vals[s] != base + cyc[s]) old_m++;

        int nvv[4];
        for (int s = 0; s < 4; s++)
            nvv[(s + face_k) % 4] = vals[s];

        int new_m = 0;
        for (int s = 0; s < 4; s++)
            if (nvv[s] != base + cyc[s]) new_m++;

        delta += (new_m - old_m);
    }

    if (heap_buf) delete[] heap_buf;
    if (heap_vis) delete[] heap_vis;
    return delta;
}


static int delta_misplaced(const SparseCube* sc, int ax, int idx, int d) {
    int n = sc->n;

    // Build cycles on stack — no heap allocation
    Cycle4 stack_cycles[1024];
    Cycle4* cycles = stack_cycles;
    Cycle4* heap_cycles = nullptr;
    if (n > 1024) {
        heap_cycles = new Cycle4[n];
        cycles = heap_cycles;
    }

    int ncyc = build_slice_cycles(n, ax, idx, cycles);
    int delta = delta_misplaced_slice_only(sc, n, cycles, ncyc, d);

    if (heap_cycles) delete[] heap_cycles;

    // Face rotation delta
    int face_fi = -1, face_k = 0;
    if (ax == 0) {
        if (idx == 0) { face_fi = 1; face_k = ((-d) % 4 + 4) % 4; }
        else if (idx == n-1) { face_fi = 0; face_k = ((d) % 4 + 4) % 4; }
    } else if (ax == 1) {
        if (idx == 0) { face_fi = 3; face_k = ((-d) % 4 + 4) % 4; }
        else if (idx == n-1) { face_fi = 2; face_k = ((d) % 4 + 4) % 4; }
    } else {
        if (idx == 0) { face_fi = 5; face_k = ((d) % 4 + 4) % 4; }
        else if (idx == n-1) { face_fi = 4; face_k = ((d) % 4 + 4) % 4; }
    }

    delta += delta_face_rotation(sc, n, face_fi, face_k);
    return delta;
}


// ══════════════════════════════════════════════════════════════
// SOLVER — with pre-allocated buffers, pass by reference
// ══════════════════════════════════════════════════════════════

struct MoveRec {
    int ax, idx, d;
};

struct SolveResult {
    MoveRec* moves;
    int nmoves;
    int* vlog_idx;
    int* vlog_dir;
    int nvlog;
    int final_misplaced;
    int solved;
};

// Process a single slice — pass vectors BY REFERENCE, not by value
static void process_slice_sparse(SparseCube* c, int ax, int idx,
                                 std::vector<MoveRec>& log,
                                 std::vector<int>& vlog_idx,
                                 std::vector<int>& vlog_dir,
                                 bool is_var_slice) {
    int cur = scube_misplaced(c);
    int best_d = 0, best_m = cur;

    // Try CW and CCW — zero-copy delta computation
    for (int d : {1, -1}) {
        int dm = delta_misplaced(c, ax, idx, d);
        int new_m = cur + dm;
        if (new_m < best_m) {
            best_m = new_m;
            best_d = d;
        }
    }

    // Try half-turn: must copy cube for 2-step look-ahead
    // But use our fast memcpy-based copy
    {
        SparseCube* t = scube_copy(c);
        scube_rot(t, ax, idx, 1);
        int dm2 = delta_misplaced(t, ax, idx, 1);
        int m1 = scube_misplaced(t);
        int m2 = m1 + dm2;
        scube_destroy(t);
        if (m2 < best_m) {
            best_m = m2;
            best_d = 2;
        }
    }

    if (best_d != 0) {
        if (best_d == 2) {
            scube_rot(c, ax, idx, 1);
            scube_rot(c, ax, idx, 1);
            log.push_back({ax, idx, 1});
            log.push_back({ax, idx, 1});
        } else {
            scube_rot(c, ax, idx, best_d);
            log.push_back({ax, idx, best_d});
        }
        if (is_var_slice) {
            vlog_idx.push_back(idx);
            vlog_dir.push_back(best_d);
        }
    }
}


SolveResult* solver_solve(SparseCube* c) {
    SolveResult* res = new SolveResult;
    int n = c->n;

    // Pre-allocate with reasonable capacity to avoid reallocs
    std::vector<MoveRec> log;
    std::vector<int> vlog_idx, vlog_dir;
    log.reserve(n * 6);
    vlog_idx.reserve(n);
    vlog_dir.reserve(n);

    if (!scube_solved(c)) {
        int half = (n + 1) / 2;

        for (int layer = 0; layer < half; layer++) {
            if (scube_solved(c)) break;
            int lo = layer;
            int hi = n - 1 - layer;

            for (int pass_idx = 0; pass_idx < (lo == hi ? 1 : 2); pass_idx++) {
                int idx = (pass_idx == 0) ? lo : hi;
                process_slice_sparse(c, 0, idx, log, vlog_idx, vlog_dir, true);
                process_slice_sparse(c, 1, idx, log, vlog_idx, vlog_dir, false);
                process_slice_sparse(c, 2, idx, log, vlog_idx, vlog_dir, false);
            }
        }

        // Correction passes
        bool improved = true;
        int max_passes = 2 * n;
        for (int pass = 0; pass < max_passes && improved && !scube_solved(c); pass++) {
            improved = false;
            for (int ax = 0; ax < 3; ax++) {
                for (int idx = 0; idx < n; idx++) {
                    if (scube_solved(c)) break;
                    int best_d = 0, best_dm = 0;
                    for (int d : {1, -1}) {
                        int dm = delta_misplaced(c, ax, idx, d);
                        if (dm < best_dm) {
                            best_dm = dm;
                            best_d = d;
                        }
                    }
                    if (best_d != 0) {
                        scube_rot(c, ax, idx, best_d);
                        log.push_back({ax, idx, best_d});
                        improved = true;
                    }
                }
            }
        }

        // Two-move corrections
        if (!scube_solved(c)) {
            bool found = true;
            int max_iter = std::min(n * 6, 500);
            for (int iter = 0; iter < max_iter && found && !scube_solved(c); iter++) {
                found = false;
                for (int a1 = 0; a1 < 3 && !found; a1++)
                    for (int i1 = 0; i1 < n && !found; i1++)
                        for (int d1 : {1, -1}) {
                            if (found) break;
                            int dm1 = delta_misplaced(c, a1, i1, d1);
                            if (dm1 > n) continue;

                            SparseCube* t = scube_copy(c);
                            scube_rot(t, a1, i1, d1);
                            for (int a2 = 0; a2 < 3 && !found; a2++)
                                for (int i2 = 0; i2 < n && !found; i2++) {
                                    if (a1==a2 && i1==i2) continue;
                                    for (int d2 : {1, -1}) {
                                        if (found) break;
                                        int dm2 = delta_misplaced(t, a2, i2, d2);
                                        if (dm1 + dm2 < 0) {
                                            scube_rot(c, a1, i1, d1);
                                            scube_rot(c, a2, i2, d2);
                                            log.push_back({a1,i1,d1});
                                            log.push_back({a2,i2,d2});
                                            found = true;
                                        }
                                    }
                                }
                            scube_destroy(t);
                        }
            }
        }
    }

    // Copy results out — move semantics for vectors would be ideal
    // but we're crossing extern "C" boundary, so we copy once
    res->nmoves = (int)log.size();
    res->moves = new MoveRec[std::max(res->nmoves, 1)];
    if (res->nmoves > 0)
        memcpy(res->moves, log.data(), res->nmoves * sizeof(MoveRec));

    res->nvlog = (int)vlog_idx.size();
    res->vlog_idx = new int[std::max(res->nvlog, 1)];
    res->vlog_dir = new int[std::max(res->nvlog, 1)];
    if (res->nvlog > 0) {
        memcpy(res->vlog_idx, vlog_idx.data(), res->nvlog * sizeof(int));
        memcpy(res->vlog_dir, vlog_dir.data(), res->nvlog * sizeof(int));
    }

    res->final_misplaced = scube_misplaced(c);
    res->solved = scube_solved(c);
    return res;
}

void result_destroy(SolveResult* r) {
    if (r) {
        delete[] r->moves;
        delete[] r->vlog_idx;
        delete[] r->vlog_dir;
        delete r;
    }
}


// ── SAT evaluation — const correctness ──

int sat_count(int nv, int nc, const int* clause_offsets,
              const int* clause_lits, int total_lits, const int* assignment) {
    int cnt = 0;
    for (int j = 0; j < nc; j++) {
        int start = clause_offsets[j];
        int end = (j+1 < nc) ? clause_offsets[j+1] : total_lits;
        bool sat = false;
        for (int k = start; k < end; k++) {
            int l = clause_lits[k];
            int v = l > 0 ? l : -l;
            if (v < 1 || v > nv) continue;
            bool val = assignment[v-1] != 0;
            if ((l > 0 && val) || (l < 0 && !val)) { sat = true; break; }
        }
        if (sat) cnt++;
    }
    return cnt;
}

int sat_eval(int nv, int nc, const int* clause_offsets,
             const int* clause_lits, int total_lits, const int* assignment) {
    return sat_count(nv,nc,clause_offsets,clause_lits,total_lits,assignment)==nc ? 1 : 0;
}

// ── Progress reporting ──
int encoding_progress = 0;
int solver_phase = 0;
int solver_layer = 0;

int get_encoding_progress() { return encoding_progress; }
int get_solver_phase() { return solver_phase; }
int get_solver_layer() { return solver_layer; }

} // extern "C"
"""

print("⚙️ Compiling C++ core (optimized: flat hashmap, zero-copy delta, release build)...")
_cpp_path = os.path.join(tempfile.gettempdir(), "rubiksat_v4.cpp")
_so_path = os.path.join(tempfile.gettempdir(), "rubiksat_v4.so")
with open(_cpp_path, 'w') as f:
    f.write(CPP_SOURCE)

# RELEASE BUILD: -O3 -march=native -flto -DNDEBUG -fomit-frame-pointer
# These are the flags that matter for "release build" performance:
# -O3: full optimization including vectorization
# -march=native: use all CPU features available
# -flto: link-time optimization (cross-function inlining)
# -DNDEBUG: disable assertions
# -fomit-frame-pointer: free up a register
# -funroll-loops: unroll small loops
subprocess.check_call([
    "g++", "-O3", "-march=native", "-flto", "-DNDEBUG",
    "-fomit-frame-pointer", "-funroll-loops",
    "-shared", "-fPIC", "-std=c++17",
    "-o", _so_path, _cpp_path
])
_lib = ctypes.CDLL(_so_path)
print("✅ C++ compiled & loaded (flat hashmap, release build).\n")

# ── C function signatures ──
_lib.scube_create.restype = ctypes.c_void_p
_lib.scube_create.argtypes = [ctypes.c_int]
_lib.scube_copy.restype = ctypes.c_void_p
_lib.scube_copy.argtypes = [ctypes.c_void_p]
_lib.scube_destroy.restype = None
_lib.scube_destroy.argtypes = [ctypes.c_void_p]
_lib.scube_solved.restype = ctypes.c_int
_lib.scube_solved.argtypes = [ctypes.c_void_p]
_lib.scube_misplaced.restype = ctypes.c_int
_lib.scube_misplaced.argtypes = [ctypes.c_void_p]
_lib.scube_rot.restype = None
_lib.scube_rot.argtypes = [ctypes.c_void_p, ctypes.c_int, ctypes.c_int, ctypes.c_int]

_lib.encode_sat.restype = ctypes.c_void_p
_lib.encode_sat.argtypes = [
    ctypes.c_int, ctypes.c_int,
    ctypes.POINTER(ctypes.c_int), ctypes.POINTER(ctypes.c_int),
    ctypes.c_int, ctypes.c_int
]
_lib.solver_solve.restype = ctypes.c_void_p
_lib.solver_solve.argtypes = [ctypes.c_void_p]
_lib.result_destroy.restype = None
_lib.result_destroy.argtypes = [ctypes.c_void_p]
_lib.sat_count.restype = ctypes.c_int
_lib.sat_count.argtypes = [
    ctypes.c_int, ctypes.c_int,
    ctypes.POINTER(ctypes.c_int), ctypes.POINTER(ctypes.c_int),
    ctypes.c_int, ctypes.POINTER(ctypes.c_int)
]
_lib.sat_eval.restype = ctypes.c_int
_lib.sat_eval.argtypes = [
    ctypes.c_int, ctypes.c_int,
    ctypes.POINTER(ctypes.c_int), ctypes.POINTER(ctypes.c_int),
    ctypes.c_int, ctypes.POINTER(ctypes.c_int)
]

# ── Structs ──
class MoveRec(ctypes.Structure):
    _fields_ = [("ax",ctypes.c_int),("idx",ctypes.c_int),("d",ctypes.c_int)]

class SolveResult(ctypes.Structure):
    _fields_ = [
        ("moves", ctypes.POINTER(MoveRec)),
        ("nmoves", ctypes.c_int),
        ("vlog_idx", ctypes.POINTER(ctypes.c_int)),
        ("vlog_dir", ctypes.POINTER(ctypes.c_int)),
        ("nvlog", ctypes.c_int),
        ("final_misplaced", ctypes.c_int),
        ("solved", ctypes.c_int),
    ]

# ══════════════════════════════════════════════════════════════
# PYTHON CLASSES
# ══════════════════════════════════════════════════════════════

class SAT:
    def __init__(self, nv: int, clauses: List[List[int]]):
        self.nv, self.clauses, self.nc = nv, clauses, len(clauses)
        all_lits, offsets = [], []
        for c in clauses:
            offsets.append(len(all_lits))
            all_lits.extend(c)
        self._offsets = (ctypes.c_int * max(len(offsets),1))(*offsets)
        self._lits = (ctypes.c_int * max(len(all_lits),1))(*all_lits)
        self._total_lits = len(all_lits)

    def count_sat(self, a: Dict[int,bool]) -> int:
        assign = (ctypes.c_int * self.nv)(*[1 if a.get(i+1,False) else 0 for i in range(self.nv)])
        return _lib.sat_count(self.nv, self.nc, self._offsets, self._lits, self._total_lits, assign)

    def eval(self, a: Dict[int,bool]) -> bool:
        assign = (ctypes.c_int * self.nv)(*[1 if a.get(i+1,False) else 0 for i in range(self.nv)])
        return _lib.sat_eval(self.nv, self.nc, self._offsets, self._lits, self._total_lits, assign) == 1


class Encoder:
    """§3: SAT → Cube configuration C_φ (sparse representation)."""
    def __init__(self, sat: SAT):
        self.sat = sat
        self.dim = 2 * (3 * sat.nv + sat.nc) + 2
        if self.dim < 6: self.dim = 6
        if self.dim % 2 != 0: self.dim += 1
        self.vs = {i: (2*i, 2*i - 1) for i in range(1, sat.nv + 1)}
        self.cs = {j: 2*sat.nv + 2*(j+1) for j in range(sat.nc)}

    def encode(self) -> ctypes.c_void_p:
        return _lib.encode_sat(
            self.sat.nv, self.sat.nc,
            self.sat._offsets, self.sat._lits, self.sat._total_lits,
            self.dim
        )


class Extractor:
    """§5: Extract σ_S from solving sequence S."""
    def __init__(self, enc: Encoder):
        self.enc = enc

    def extract(self, seq: list, vlog: dict) -> Dict[int, bool]:
        a = {}
        for vi, (sp, sn) in self.enc.vs.items():
            if sp in vlog:
                a[vi] = (vlog[sp] == 1)
            elif sn in vlog:
                a[vi] = (vlog[sn] == 1)
            else:
                nr = sum(d for ax, i, d in seq if ax == 0 and i == sp)
                a[vi] = (nr > 0)
        for i in range(1, self.enc.sat.nv + 1):
            if i not in a:
                a[i] = True
        return a


# ══════════════════════════════════════════════════════════════
# DIMACS PARSER
# ══════════════════════════════════════════════════════════════

def parse_dimacs(text: str) -> SAT:
    nv = 0; clauses = []; buf = []
    for line in text.split('\n'):
        line = line.strip()
        if not line or line[0] in ('c','%'): continue
        if line[0] == 'p':
            p = line.split()
            if len(p) >= 4: nv = int(p[2])
            continue
        for tok in line.split():
            try: val = int(tok)
            except ValueError: continue
            if val == 0:
                if buf: clauses.append(buf); buf = []
            else: buf.append(val)
    if buf: clauses.append(buf)
    if nv == 0 and clauses: nv = max(abs(l) for c in clauses for l in c)
    return SAT(nv, clauses)

def read_file(path: str) -> str:
    low = path.lower()
    if low.endswith('.xz') or low.endswith('.lzma'):
        import lzma
        with lzma.open(path,'rt',encoding='ascii',errors='replace') as f: return f.read()
    elif low.endswith('.gz'):
        import gzip
        with gzip.open(path,'rt',encoding='ascii',errors='replace') as f: return f.read()
    elif low.endswith('.bz2'):
        import bz2
        with bz2.open(path,'rt',encoding='ascii',errors='replace') as f: return f.read()
    else:
        with open(path,'r',encoding='ascii',errors='replace') as f: return f.read()

def cstr(c):
    return "(" + " ∨ ".join(f"x{abs(l)}" if l>0 else f"¬x{abs(l)}" for l in c) + ")"

# ══════════════════════════════════════════════════════════════
# PIPELINE
# ══════════════════════════════════════════════════════════════

AX_NAME = {0:'x', 1:'y', 2:'z'}

def run_pipeline(sat: SAT):
    log = []
    L = lambda *a: log.append(" ".join(str(x) for x in a))
    st = {}

    L("═"*60)
    L("PHASE 1: ENCODING φ → C_φ (Section 3)")
    L("═"*60)
    t0 = time.time()
    enc = Encoder(sat)
    print(f"  Encoding: N={enc.dim}, starting...")
    cube_ptr = enc.encode()
    t1 = time.time()
    st['dim'] = enc.dim
    st['mis0'] = _lib.scube_misplaced(cube_ptr)
    st['t_enc'] = t1 - t0

    L(f"  §3.1: N = 2(3·{sat.nv} + {sat.nc}) + 2 = {enc.dim}")
    L(f"  N³ = {enc.dim**3:,} (cubies)")
    L(f"  6N² = {6*enc.dim**2:,} (stickers)")
    L(f"  sparse entries = {st['mis0']} (only displaced stickers tracked)")
    L(f"  sparsity = {st['mis0']/(6*enc.dim**2)*100:.4f}%")
    L(f"  time = {st['t_enc']:.6f}s")
    L()
    L("  §3.2 Variable slices (x-axis):")
    for vi in range(1, min(sat.nv+1, 21)):
        sp, sn = enc.vs[vi]
        L(f"    x{vi} → s⁺={sp} s⁻={sn}")
    if sat.nv > 20:
        L(f"    ... ({sat.nv-20} more variables)")
    L()
    L("  §3.3 Clause slices (y-axis):")
    for j in range(min(sat.nc, 10)):
        q = enc.cs[j]
        L(f"    C{j+1} {cstr(sat.clauses[j]):40s} → q={q}")
    if sat.nc > 10:
        L(f"    ... ({sat.nc-10} more clauses)")
    L()

    print(f"  Encoding done: {st['mis0']} displaced stickers ({st['t_enc']:.2f}s)")

    # ── §4: Solve ──
    L("═"*60)
    L("PHASE 2: REDUCTION METHOD O(N²) (Section 4)")
    L("═"*60)
    print(f"  Solving: {st['mis0']} misplaced stickers...")
    t2 = time.time()
    res_ptr = _lib.solver_solve(cube_ptr)
    res = SolveResult.from_address(res_ptr)
    t3 = time.time()

    seq = [(res.moves[i].ax, res.moves[i].idx, res.moves[i].d)
           for i in range(res.nmoves)]
    vlog = {}
    for i in range(res.nvlog):
        vlog[res.vlog_idx[i]] = res.vlog_dir[i]

    st['moves'] = res.nmoves
    st['solved'] = res.solved != 0
    st['mis1'] = res.final_misplaced
    st['t_sol'] = t3 - t2

    _lib.result_destroy(res_ptr)

    L(f"  moves = {st['moves']}")
    L(f"  cube solved = {'YES' if st['solved'] else 'NO'}")
    L(f"  residual misplaced = {st['mis1']}")
    L(f"  time = {st['t_sol']:.6f}s")
    L()
    show = min(len(seq), 200)
    L("  Sequence S (first moves):")
    for k in range(show):
        ax,idx,d = seq[k]
        L(f"    S[{k+1:>5}] axis={AX_NAME[ax]} slice={idx:>4} dir={'+1' if d==1 else '-1'}")
    if len(seq)>show:
        L(f"    ... ({len(seq)-show} more)")
    L()

    L("  §4.2 Variable slice decisions:")
    for idx_k in sorted(vlog):
        d = vlog[idx_k]
        for vi,(sp,sn) in enc.vs.items():
            if idx_k == sp:
                L(f"    s⁺(x{vi})={idx_k} dir={d:+d} → σ(x{vi})={'T' if d==1 else 'F'}")
                break
            elif idx_k == sn:
                L(f"    s⁻(x{vi})={idx_k} dir={d:+d} → σ(x{vi})={'T' if d==1 else 'F'}")
                break
    L()

    print(f"  Solving done: {st['moves']} moves, solved={'YES' if st['solved'] else 'NO'} ({st['t_sol']:.2f}s)")

    # ── §5: Extract ──
    L("═"*60)
    L("PHASE 3: EXTRACTION S → σ_S (Section 5)")
    L("═"*60)
    t4 = time.time()
    ext = Extractor(enc)
    sigma = ext.extract(seq, vlog)
    t5 = time.time()
    st['t_ext'] = t5 - t4

    L(f"  time = {st['t_ext']:.6f}s")
    L()
    L("  σ_S assignment:")
    for vi in range(1, sat.nv+1):
        v = sigma.get(vi, False)
        L(f"    x{vi} = {1 if v else 0}")
    L()

    # ── Verify ──
    L("═"*60)
    L("PHASE 4: VERIFICATION (Section 5.3)")
    L("═"*60)

    cnt = sat.count_sat(sigma)
    ok = sat.eval(sigma)
    st['cnt'] = cnt
    st['sat'] = ok

    L(f"  clauses satisfied = {cnt}/{sat.nc}")
    L(f"  φ(σ_S) = {'1 ✓ SATISFIABLE' if ok else '0 ✗ NOT ALL SATISFIED'}")
    L()

    t6 = time.time()
    st['t_total'] = t6 - t0

    L("═"*60)
    L("RESULT")
    L("═"*60)
    L(f"  {'SATISFIABLE' if ok else 'UNSATISFIABLE'}")
    if ok:
        vals = " ".join(str(vi) if sigma.get(vi,False) else str(-vi) for vi in range(1,sat.nv+1))
        L(f"  v {vals} 0")
    L()
    L("─"*60)
    L("STATISTICS")
    L("─"*60)
    L(f"  variables          {sat.nv}")
    L(f"  clauses            {sat.nc}")
    L(f"  cube dimension     {st['dim']} (N=2(3n+m)+2)")
    L(f"  sparse entries     {st['mis0']} ({st['mis0']/(6*st['dim']**2)*100:.4f}% of 6N²)")
    L(f"  final misplaced    {st['mis1']}")
    L(f"  moves              {st['moves']}")
    L(f"  cube solved        {'yes' if st['solved'] else 'no'}")
    L(f"  clauses sat        {st['cnt']}/{sat.nc}")
    L(f"  t_encode           {st['t_enc']:.6f}s")
    L(f"  t_solve            {st['t_sol']:.6f}s")
    L(f"  t_extract          {st['t_ext']:.6f}s")
    L(f"  t_total            {st['t_total']:.6f}s")
    L("═"*60)

    _lib.scube_destroy(cube_ptr)
    return ok, sigma, st, log


def write_dynamics(path, input_name, sat, ok, sigma, st, log):
    with open(path, 'w', encoding='utf-8') as f:
        f.write(f"rubiksat — {input_name}\n")
        f.write(f"date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"vars={sat.nv} clauses={sat.nc} cube={st['dim']}³\n\n")
        for line in log:
            f.write(line + '\n')

# ══════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════

USE_UPLOAD = True #@param {type:"boolean"}

if USE_UPLOAD:
    print("📂 Sube tu archivo .cnf / .cnf.xz / .cnf.gz / .cnf.bz2\n")
    uploaded = colab_files.upload()
    if not uploaded:
        raise SystemExit("No file uploaded.")
    fname = list(uploaded.keys())[0]
    text = read_file(fname)
else:
    fname = "example.cnf"
    text = """c example SAT instance
p cnf 3 4
1 2 -3 0
-1 3 0
2 -3 0
-1 -2 3 0
"""

sat = parse_dimacs(text)
enc_preview = Encoder(sat)
dim = enc_preview.dim

print(f"{'═'*60}")
print(f" rubiksat — SAT via Rubik's Cube (Optimized, C++ accel)")
print(f" File:      {fname}")
print(f" Variables: {sat.nv}")
print(f" Clauses:   {sat.nc}")
print(f" §3.1 N = {dim}")
print(f" Cube:      {dim}³ = {dim**3:,} cubies")
print(f" 6N² = {6*dim**2:,} stickers (sparse: only displaced tracked)")
print(f"{'═'*60}\n")

if sat.nc == 0:
    print("s SATISFIABLE (trivial: 0 clauses)")
    raise SystemExit()

ok, sigma, stats, log = run_pipeline(sat)

print()
if ok:
    print("s SATISFIABLE")
    vals = " ".join(str(vi) if sigma.get(vi,False) else str(-vi) for vi in range(1,sat.nv+1))
    print(f"v {vals} 0")
else:
    print("s UNSATISFIABLE")

print(f"\nc {stats['cnt']}/{sat.nc} clauses | cube {stats['dim']}³ | "
      f"{stats['moves']} moves | sparse {stats['mis0']} entries | {stats['t_total']:.4f}s")

base = fname
for ext in ('.xz','.lzma','.gz','.bz2'):
    if base.lower().endswith(ext): base = base[:-len(ext)]
if base.lower().endswith('.cnf'): base = base[:-4]
out_path = f"{base}_dynamics.txt"

write_dynamics(out_path, fname, sat, ok, sigma, stats, log)
print(f"c dynamics → {out_path}")

print(f"\n{'═'*60}")
print(f" {out_path}")
print(f"{'═'*60}\n")
for line in log:
    print(line)

try:
    colab_files.download(out_path)
except: pass